In [1]:
import pickle
import sys
import copy
import time

import cobra

import multiprocessing
from multiprocessing import Process

sys.path.insert(1, '../../scripts/')
from utils import functions as func

In [2]:
lp_path = '/data2/hratch/human_me/test_lp/'

with open(lp_path + 'me_model.pickle', 'rb') as handle:
    me_model = pickle.load(handle)

with open(lp_path + 'final_reactions.pickle', 'rb') as handle:
    final_reactions = pickle.load(handle)

In [14]:
def glpk(mu_val):
    final_reactions_ = copy.deepcopy(final_reactions)
    for r in final_reactions_:
        if isinstance(r, func.ME_Reaction):
            if 'biomass' not in r.type:
                r.replace_coefficient_mu(mu_val = mu_val)
            else:
                r.replace_bound_mu(mu_val = mu_val, inplace = True)

    glpk_model = cobra.Model('glpk')
    glpk_model.add_reactions(final_reactions)
    glpk_model.objective = {glpk_model.reactions.biomass_dilution: 1}
    
    try:
        start = time.time()
        res = glpk_model.optimize()
        end = time.time()
        tot = str((end-start)/3600)
        
        with open(lp_path + 'optimization_outputs.tab', 'a') as f:
            f.write(str(mu_val) + '\t' + 'GLPK' + '\t' + tot + '\t' + '' + '\n')
        
        
        res = res.to_frame()
        res.to_csv(lp_path + 'glpk_' + str(mu_val).replace('.', '_') + '.csv')
    except:
        with open(lp_path + 'optimization_outputs.tab', 'a') as f:
            f.write(str(mu_val) + '\t' + 'GLPK' + '\t' + 'failed' + '\t' + 'failed' + '\n')

def qminos(mu_val, precision):
    start = time.time()
    xq,statq,hsq = me_model.solve_lp(mu_val = mu_val, precision = precision)
    end = time.time()
    tot = str((end-start)/3600)
    
    with open(lp_path + 'optimization_outputs.tab', 'a') as f:
        f.write(str(mu_val) + '\t' + 'QMINOS_' + precision + '\t' + tot + '\t' + str(statq.var()) + '\n')
    
    res = pd.DataFrame(xq)
    res.to_csv(lp_path + 'qminos_' + precision + '_' + str(mu_val).replace('.', '_') + '.csv')

def qminos_dual(mu_val):
    qminos(mu_val, precision = 'double')
    
def qminos_quad(mu_val):
    qminos(mu_val, precision = 'quad')
    
def runSimultaneously(mu_val): #,*fns):
    proc = []
    fns = [glpk, qminos_dual, qminos_quad]
    for fn in fns:
        p = Process(target=fn, args = (mu_val,))
        p.start()
        proc.append(p)
    for p in proc:
        p.join()        

In [ ]:
with open(lp_path + 'optimization_outputs.tab', 'w') as f:
    f.write('mu' + '\t' + 'Algorithm' + '\t' + 'Run Time' + '\t' + 'Status' + '\n')

mu_vals = [1e-6, 1e-3, 5e-3, 1e-2, 2e-2, 3e-2]
n_cores = len(mu_vals)

pool = multiprocessing.Pool(processes=n_cores)
pool.map(runSimultaneously, mu_vals)        
pool.close()